In [ ]:
from glob import glob
from cmcmultimodal.io import mat2nii
from pathlib import Path
import os

my_files = glob('/Users/Vasilis/Downloads/Zebel/PSOCT/Retardance/Slice*.mat')
highres_files = []
lowres_files = []
for f in my_files:
    nii_highres, nii_lowres = mat2nii(f, nii_lowres_file=os.path.join(Path(f).parent,'lowres/',
                                      Path(f).name.replace('.mat','.nii.gz')), downsample=10)
    highres_files.append(nii_highres)
    lowres_files.append(nii_lowres)
    
highres_files

In [ ]:
#!/bin/bash

python psoct_wrapper.py --inp_path '/Users/Vasilis/CMC_data/Moe' --out_path '/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret' --seq_params '/Users/Vasilis/CMC_data/Moe/PSOCT/seq_params.json' --mri_ref '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz' --reg_modality Retardance --reg_method 'cc' --other_images Retardance Orientation --bad_slides 140 --verbose

In [4]:
# Process all MOE slides

from cmcmultimodal.proc import psoct
from pathlib import Path

datadir = '/Users/Vasilis/CMC_data/Moe'
output_path = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross'
mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
seq_params = '/Users/Vasilis/CMC_data/Moe/PSOCT/seq_params.json'

my_data = psoct(Path(datadir), seq_params, reg_modality='Cross', verbose=True)

indiv_slides = my_data.run_pipeline(other_images=['Retardance'], 
                                    output_path=output_path, 
                                    mri_ref=mri_ref, 
                                    downsample=1, 
                                    bad_slides=[140,],
                                    reg_method='flirt')


Reading input information for '/Users/Vasilis/CMC_data/Moe' ...
	Input folder read successfully.
	PSOCT sequence parameters read successfully.
	Found 8 missing slides: [99, 100, 229, 105, 106, 107, 108, 109]

Starting slide registration process ...
	Missing slides have been interpolated successfully.
	Reference slide for alignment: 184, size=(1100, 874)
Slide registration completed.

Starting slide deck creation ...
	Relative and absolute shifts saved.
	Creating slide deck image ... Done.
	Running MRI-to-PSOCT registration ... Done.
	Updating headers for registration slides ...  Done.
	Applying registration matrix to 'Retardance' slides ...
		Unexpected number of matching files: file number 229. Skipping this file.
		Unexpected number of matching files: file number 109. Skipping this file.
		Unexpected number of matching files: file number 108. Skipping this file.
		Unexpected number of matching files: file number 107. Skipping this file.
		Unexpected number of matching files: file nu

In [ ]:
from pathlib import Path
from cmcmultimodal.validate_results import compare_results_folder

ref_path = Path('/Users/Vasilis/Downloads/CMC_results/old/Moe_flirt_Cross')
est_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross')

compare_results_folder(ref_path, est_path, subdir=False)

/Users/Vasilis/Downloads/CMC_results/old/Moe_flirt_Cross_test/PSOCT_Ret_in_DTI.nii.gz
	 File does not exist in estimated path!
/Users/Vasilis/Downloads/CMC_results/old/Moe_flirt_Cross_test/mri_to_slides.mat
	 File does not exist in estimated path!
/Users/Vasilis/Downloads/CMC_results/old/Moe_flirt_Cross_test/slides_to_mri.mat
	 File does not exist in estimated path!
/Users/Vasilis/Downloads/CMC_results/old/Moe_flirt_Cross_test/slide_deck_to_mri.nii.gz
	 File does not exist in estimated path!
/Users/Vasilis/Downloads/CMC_results/old/Moe_flirt_Cross_test/reoriented_FA_to_slides.nii.gz
	 File does not exist in estimated path!


In [ ]:
from fsl.wrappers import fnirt
import subprocess

mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
deck = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/Cro_slide_deck.nii.gz'
outfile = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/Cro_slide_deck_in_MRI_fnirt'
affmat = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/PSOCT_to_MRI.mat'
config = '../data/config'

# flirt(src=mri_img, ref=self.slide_deck_img, out=outfile, omat=matfile, dof=12, interp='spline')

subprocess.run(["fnirt",
                f"--in={deck}",
                f"--iout={outfile}",
                f"--ref={mri_ref}",
                f"--aff={affmat}",
                f"--config={config}",
                "--verbose",
])



Part of FSL (ID: "")
fnirt

Usage: 
fnirt --ref=<some template> --in=<some image>
fnirt --ref=<some template> --in=<some image> --infwhm=8,4,2 --subsamp=4,2,1 --warpres=8,8,8

Compulsory arguments (You MUST set one or more of):
	--ref		name of reference image
	--in		name of input image

Optional arguments (You may optionally specify one or more of):
	--aff		name of file containing affine transform
	--inwarp	name of file containing initial non-linear warps
	--intin		name of file/files containing initial intensity mapping
	--cout		name of output file with field coefficients
	--iout		name of output image
	--fout		name of output file with field
	--jout		name of file for writing out the Jacobian of the field (for diagnostic or VBM purposes)
	--refout	name of file for writing out intensity modulated --ref (for diagnostic purposes)
	--intout	name of files for writing information pertaining to intensity mapping
	--logout	Name of log-file
	--config	Name of config file specifying command line a

CompletedProcess(args=['fnirt', '--in=', '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/Cro_slide_deck.nii.gz', '--iout=', '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/Cro_slide_deck_in_MRI_fnirt', '--ref=', '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz', '--aff=', '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/PSOCT_to_MRI.mat', '--config=', '../data/config', '--verbose'], returncode=1)

In [ ]:
from fsl.wrappers.avwutils  import fslmerge
import glob


out_file = '/Users/Vasilis/CMC_results/Moe_flirt_Cross/Ret_slide_deck_in_mri_2'
inp_files = sorted(glob.glob('/Users/Vasilis/CMC_results/Moe_flirt_Cross/Retardance/lowres/Slice*'), reverse=True)

fslmerge('y', out_file, *inp_files[98:246])

{}

In [4]:
from fsl.wrappers.avwutils  import fslmerge
import glob


out_file = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_test/PSOCT_Ret_in_DTI'
inp_files = sorted(glob.glob('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_test/Retardance/lowres/Slice*'), reverse=True)


fslmerge('y', out_file, *inp_files)

{}

In [28]:
import json
import numpy as np
from pathlib import Path

output_path = Path('/Users/Vasilis/Downloads/PSOCT_all_slides_Cross_7')

with open(output_path / "abs_shifts.json") as f:
    data = json.load(f)

abs_shifts = {int(k): np.array(v) for k, v in data.items()}

In [17]:
from fsl.wrappers import flirt
from cmcmultimodal.utils import pad_image, save_nifti
from fsl.data.image import Image
import numpy as np

out_path=Path('/Users/Vasilis/Downloads/PSOCT_all_slides_Mixed_2/Retardance')
inp_path=Path('/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/Retardance')
ref_path=Path('/Users/Vasilis/Downloads/PSOCT_all_slides_Mixed_2/Mixed/lowres')

i = 180

ref = ref_path / ('Slice_' + str(i) + '_EnRCr_hdr')
src = inp_path / ('Slice_' + str(i) + '_EnR')

img_highres = Image(src).data[:,:,0]
highres_shape = [x * my_data.downsample for x in my_data.ref_shape]
img_padded    = pad_image(img_highres, highres_shape)
src_filename  = 'tmp_padded_source.nii.gz'
save_nifti(img_padded, src_filename)

mat = np.loadtxt(out_path / '../slides_to_mri.mat')

T = mat * my_data.abs_shifts[i]

img_padded = flirt(src_filename, ref, init=T, out=out_path / (src.name + '_hdr'), twod=True, applyxfm=True)


In [14]:
highres_shape

[11000, 8740]

In [15]:
img_padded

{}

In [ ]:
from fsl.wrappers import applyxfm, flirt
from cmcmultimodal.utils import pad_image
from fsl.data.image import Image
from cmcmultimodal.io import save_nifti

out_path=Path('/Users/Vasilis/Downloads/PSOCT_test/')
inp_path=Path('/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/')
ref_path=Path('/Users/Vasilis/Downloads/PSOCT_all_slides_Cross_7')

# i = 150
for i in abs_shifts.keys():
    inp = inp_path / 'Retardance' / 'lowres' / ('Slice_' + str(i).zfill(3) +'_EnR.nii.gz')
    # inp = inp_path / 'Cross' / 'lowres' / ('Slice_' + str(i) +'_EnCr.nii.gz')
    ref = ref_path / 'Cross' / 'lowres' / 'Slice_184_EnCr_hdr.nii.gz'
    try:
        inp_image = Image(inp).data[:,:,0]
    except:
        continue
    # shape = Image(ref).data[:,0,:].shape
    shape=(1100, 874)

    src_padded = pad_image(inp_image, shape)
    src_filename = out_path / 'padded_source'
    save_nifti(src_padded, src_filename)

    # applyxfm(src_filename, ref, abs_shifts[i], out_path / inp.name, twod=True)
    flirt(src_filename, ref, init=abs_shifts[i], out=out_path / inp.name, twod=True, applyxfm=True)



In [ ]:
from fsl.wrappers import flirt, LOAD, applyxfm
from cmcmultimodal.utils import pad_image
from cmcmultimodal.io import save_nifti
import numpy as np
import os
from fsl.data.image         import Image

# inp_path='/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/Retardance/lowres/'
inp_path='/Users/Vasilis/Downloads/PSOCT_Ret/PSOCT/Cross/lowres/'
out_path='/Users/Vasilis/Downloads/PSOCT_test/'
os.makedirs(out_path, exist_ok=True)
for i in [160]:#range(98,201):
    # inp='Slice_' + str(i) +'_EnR.nii.gz'
    # ref='Slice_325_EnR.nii.gz'
    inp='Slice_' + str(i) +'_EnCr.nii.gz'
    ref='Slice_161_EnCr.nii.gz'
    # shape=(1100, 874)
    # # shape = [i+min(shape)//10 for i in shape]
    try:
        src_data = Image(inp_path+inp).data[:,:,0]
    except:
        continue
    tgt_data = Image(inp_path+ref).data[:,:,0]
    shape = tuple(map(max, zip(src_data.shape, tgt_data.shape)))
    # increase max shape by 10% each side
    shape = tuple(i+min(shape)//5 for i in shape)
    src_padded = pad_image(src_data, shape)
    tgt_padded = pad_image(tgt_data, shape)
    src_filename = out_path + 'padded_source'
    tgt_filename = out_path + 'padded_target'
    save_nifti(src_padded, src_filename)
    save_nifti(tgt_padded, tgt_filename)
    t = flirt(src_filename, tgt_filename,
                omat=LOAD,# out= out_path + inp,
                cost='leastsq',
                twod=True)
    # cost='normcorr' or 'leastsq'
    # t= flirt(src_filename, tgt_filename,
    #      omat='tmpmat1.mat',
    #      out='tmpfile1', cost='leastsq',
    #      twod=True)
    # t = np.loadtxt('tmpmat1.mat')
    print(t['omat'])
    img_padded = applyxfm(src_filename, tgt_filename, t['omat'], out_path + inp, twod=True)
    # img_padded = flirt(src_filename, tgt_filename, init=t['omat'], out=out_path + 'Try2_' + inp, applyxfm=True,
                # )
    # img_padded['out'].get_fdata()
# print(img_padded)
# flirt(src_filename, tgt_filename,
#      omat='tmpmat2.mat',
#      out='tmpfile2', cost='normcorr',
#      twod=True)
# t = np.loadtxt('tmpmat2.mat')
# print(t)

[[ 1.00000000e+00  1.89414713e-05  0.00000000e+00  4.53207593e-01]
 [-1.89414713e-05  1.00000000e+00  0.00000000e+00  1.47949071e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [10]:
# test fslsplit in the three orientations
from fsl.wrappers.avwutils  import fslsplit

inp_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_3')
out_path = Path('/Users/Vasilis/Downloads/fslsplit_test')
slide_deck = inp_path / 'slide_deck.nii.gz'
orientation = 'sagittal'  

# Lookup table for orientation information
OrientationLookup = {'sagittal': 'x', 'coronal': 'y', 'axial': 'z',
                     'Sagittal': 'x', 'Coronal': 'y', 'Axial': 'z'}

indiv_slides = fslsplit(src=slide_deck, out=out_path/orientation, dim=OrientationLookup[orientation])
